In [1]:
import torch
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer

ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install an xformers build matching your PyTorch and CUDA versions for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn

In [2]:
import torch.nn.functional as F

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings


In [12]:
pcv_seq = "MPSKKNGRSGPQPHKRWVFTLNNPSEDERKKIREPPISLFDYFIVGEEGNEEGRTPHLQGFANFVKKQTFNKVKWYLGARCHIEKAKGTDQQNKEYCSKEGNLLIECGAPRSQGQRSDLSTAVSTLLESGSLVTVAEQHPVTFVRNFRGLAELLKVSGKMQKRDWKTNVHVIVGPPGCGKSKWAANFADPETTYWKPPRNKWWDGYHGEEVVVIDDFYGWLPWDDLLRLCDRYPLTVETKGGTVPFLARSILITSNQTPLEWYSSTAVPAVEALYRRITSLVFWKNATEQSTEEGGQFVTLSPPCPEFPYEINY"
wdv_seq = "MASSSAPRFRVYSKYLFLTYPECTLEPQYALDSLRTLLNKYEPLYIAAVRELHEDGSPHLHVLVQNKLRASITNPNALNLRMDTSPFSIFHPNIQAAKDCNQVRDYITKEVDSDVNTAEWGTFVAVSTPGRKDRDADMKQIIESSSSREEFLSMVCNRFPFEWSIRLKDFEYTARHLFPDPVATYTPEFPTESLICHETIESWKNEHLYSESPGRHKSIYICGPTRTGKTSWARSLGTHNYYNSLVDFTTYDVNAKYNIIDDIPFKFTPNWKCFVGAQRDFTVNPKYGKRKVIRGGIPCIILVNPDEDWLKDMTPEQSDYMYSNTVVHYMYEGETFINYSFASGEDVTASQ"
fbnyv_seq = "MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYIEMKKRTSLAGMKKLIPGAHFEKRRGTQGEARAYSMKEDTRLEGPWEYGEFVPTIEDKLREVMNDMKITGKRPIEYIEECCNTYDKSASTLREFRGELKKKKAISSWELQRKPWMGEVDALLQERDGRRIIWVYGPQGGEGKTSYAKHLVKTRDAFYSTGGKTADIAFAWDHQELVLFDFPRSFEEYVNYGVIEQLKNGIIQSGKYQSVIKYSDYVEVIVFANFTPRSGMFSEDRIVYVYA"

pcv_embed = get_protein_embedding(pcv_seq)
wdv_embed = get_protein_embedding(wdv_seq)
fbnyv_embed = get_protein_embedding(fbnyv_seq)

wdv_sDBM = wdv_embed[90:99, :]
pcv_sDBM = pcv_embed[78:87, :]
fbnyv_sDBM = fbnyv_embed[62:71, :]


In [77]:
traI_seq = """
MMSIAQVRSAGSAGNYYTDKDNYYVLGSMGERWAGRGAEQLGLQGSVDKDVFTRLLEGRL
PDGADLSRMQDGSNRHRPGYDLTFSAPKSVSMMAMLGGDKRLIDAHNQAVDFAVRQVEAL
ASTRVMTDGQSETVLTGNLVMALFNHDTSRDQEPQLHTHAVVANVTQHNGEWKTLSSDKV
GKTGFIENVYANQIAFGRLYREKLKEQVEALGYETEVVGKHGMWEMPGVPVEAFSGRSQT
IREAVGEDASLKSRDVAALDTRKSKQHVDPEIKMAEWMQTLKETGFDIRAYRDAADQRAD
LRTLTPGPASQDGPDVQQAVTQAIAGLSERKVQFTYTDVLARTVGILPPENGVIERARAG
IDEAISREQLIPLDREKGLFTSGIHVLDELSVRALSRDIMKQNRVTVHPEKSVPRTAGYS
DAVSVLAQDRPSLAIVSGQGGAAGQRERVAELVMMAREQGREVQIIAADRRSQMNMKQDE
RLSGELITGRRQLLEGMAFTPGSTVIVDQGEKLSLKETLTLLDGAARHNVQVLITDSGQR
TGTGSALMAMKDAGVNTYRWQGGEQRPATIISEPDRNVRYARLAGDFAASVKAGEESVAQ
VSGVREQAILTQAIRSELKTQGVLGLPEVTMTALSPVWLDSRSRYLRDMYRPGMVMEQWN
PETRSHDRYVIDRVTAQSHSLTLRDAQGETQVVRISSLDSSWSLFRPEKMPVADGERLRV
TGKIPGLRVSGGDRLQVASVSEDAMTVVVPGRAEPATLPVSDSPFTALKLENGWVETPGH
SVSDSATVFASVTQMAMDNATLNGLARSGRDVRLYSSLDETRTAEKLARHPSFTVVSEQI
KTRAGETSLETAISHQKSALHTPAQQAIHLALPVVESKKLAFSMVDLLTEAKSFAAEGTG
FTELGGEINAQIKRGDLLYVDVAKGYGTGLLVSRASYEAEKSILRHILEGKEAVMPLMER
VPGELMEKLTSGQRAATRMILETSDRFTVVQGYAGVGKTTQFRAVMSAVNMLPESERPRV
VGLGPTHRAVGEMRSAGVDAQTLASFLHDTQLQQRSGETPDFSNTLFLLDESSMVGNTDM
ARAYALIAAGGGRAVASGDTDQLQAIAPGQPFRLQQTRSAADVAIMKEIVRQTPELREAV
YSLINRDVERALSGLESVKPSQVPRQEGAWAPEHSVTEFSHSQEAKLAEAQQKAMLKGEA
FPDVPMTLYEAIVRDYTGRTPEAREQTLIVTHLNEDRRVLNSMIHDVREKAGELGKEQVM
VPVLNTANIRDGELRRLSTWETHRDALVLVDNVYHRIAGISKDDGLITLQDAEGNTRLIS
PREAVAEGVTLYTPDTIRVGTGDRMRFTKSDRERGYVANSVWTVTAVSGDSVTLSDGQQT
REIRPGQEQAEQHIDLAYAITAHGAQGASETFAIALEGTEGNRKLMAGFESAYVALSRMK
QHVQVYTDNRQGWTDAINNAVQKGTAHDVFEPKPDREVMNAERLFSTARELRDVAAGRAV
LRQAGLAGGDSPARFIAPGRKYPQPYVALPAFDRNGKSAGIWLNPLTTDDGNGLRGFSGE
GRVKGSGDAQFVALQGSRNGESLLADNMQDGVRIARDNPDSGVVVRIAGEGRPWNPGAIT
GGRVWGDIPDNSVQPGAGNGEPVTAEVLAQRQAEEAIRRETERRADEIVRKMAENKPDLP
DGKTEQAVREIAGQERDRAAITEREAALPEGVLREPQRVREAVREIARENLLQERLQQME
RDMVRDLQKEKTLGGD
"""


trwC_seq = """
MLSHMVLTRQDIGRAASYYEDGADDYYAKDGDASEWQGKGAEELGLSGEVDSKRFRELLA
GNIGEGHRIMRSATRQDSKERIGLDLTFSAPKSVSLQALVAGDAEIIKAHDRAVARTLEQ
AEARAQARQKIQGKTRIETTGNLVIGKFRHETSRERDPQLHTHAVILNMTKRSDGQWRAL
KNDEIVKATRYLGAVYNAELAHELQKLGYQLRYGKDGNFDLAHIDRQQIEGFSKRTEQIA
EWYAARGLDPNSVSLEQKQAAKVLSRAKKTSVDREALRAEWQATAKELGIDFSRREWSGR
EKGGSEKQAHSFMPSDEAAKRAVRYAINHLTERQSVMDERELVDTAMKHAVGAARLEDIQ
KELLRQTETGYLIREAPRYRPGGQTGPTDEPGKTRAEWVAELAAKGMKQGAARERVDNAI
KTGGLVPIEPRYTTQTALEREKRILQIERDGRGAVAPVIAAEAARERLASTNLNQGQREA
AELIVSAANRVVGVQGFAGTGKSHMLDTAKQMIEGEGYHVRALAPYGSQVKALRELNVEA
NTLASFLRAKDKNIDSRTVLVIDEAGVVPTRLMEQTLKLAEKAGARVVLMGDTAQTKAIE
AGRPFDQLQAAGMQTAHMREIQRQKNPELKIAVELAAAGKASSSLERIKDVTEIKNHHER
RAAVAEAYIALKPDERDRTLIVSGTNEARREINQIVREGLGTAGKGIEFDTLVRVDTTQA
ERRHSKNYQVGHVIQPERDYAKTGLQRGELYRVVETGPGNRLTVIGEHDGQRIQFSPMTH
TKISVYQPERAELAVGDTIRITRNDKHLDLANGDRMKVVAVEDRKVTVTDGKRNVELPTD
KPLHVDHAYATTVHSSQGLTSDRVLIDAHAESRTTAKDVYYVAISRARFEARVFTNDRGK
LPAAIARENIKSAAHDLARDRGGRSAAAERQREQQREAERNRQTQQPAHDRQKAAREAER
GMEAGR
"""

nes_seq = """
MAMYHFQNKFVSKANGQSATAKSAYNSASRIKDFKENEFKDYSNKQCDYSEILLPNNADD
KFKDREYLWNKVHDVENRKNSQVAREIIIGLPNEFDPNSNIELAKEFAESLSNEGMIVDL
NIHKINEENPHAHLLCTLRGLDKNNEFEPKRKGNDYIRDWNTKEKHNEWRKRWENVQNKH
LEKNGFSVRVSADSYKNQNIDLEPTKKEGWKARKFEDETGKKSSISKHNESIKKRNQQKI
DKMFNEVDDMKSHKLNAFSYMNKSDSTTLKNMAKDLKIYVTPINMYKENERLYDLKQKTS
LITDDEDRLNKIEDIEDRQKKLESINEVFEKQAGIFFDKNYPDQSLNYSDDEKSLSHMHY
INDRDVLPANNELEDIVKEKRIKEAQISLNTVLGNRDISLESIEKESNFFADKLSNILEK
NNLSFDDVLENKHEGMEDSLKIDYYTNKLEVFRNAENILEDYYDVQIKELFTDDEDYKAF
NEVTDIKEKQQLIDFKTYHGTENTIEMLETGNFIPKYSDEDRKYITEQVKLLQEKEFKPN
KNQHDKFVFGAIQKKLLSEYDFDYSDNNDLKHLYQESNEVGDEISKDNIEEFYEGNDILV
DKETYNNYNRKQQAYGLINAGLDSMIFNFNEIFRERMPKYINHQYKGKNHSKQRHELKNK
RGMHL
"""

tnpa_seq = """
MSNAVLYKSNHNVVYSCKYHIVWCPKYRRKVLVGAVEMRLKEIIQEVAKELRVEIIEMQT
DKDHIHILADIDPSFGVMKFIKTAKGRSSRILRQEFNHLKTKLPTLWTNSCFISTVGGAP
LNVVKQYIENQQNSNRPKQKEKWKSYVDNLQTKAL
"""

traI_seq = traI_seq.replace("\n", "")
print(traI_seq)

trwC_seq = trwC_seq.replace("\n", "")

nes_seq = nes_seq.replace("\n", "")
tnpa_seq = tnpa_seq.replace("\n", "")

traI_embed = get_protein_embedding(traI_seq)
trwC_embed = get_protein_embedding(trwC_seq)
nes_embed = get_protein_embedding(nes_seq)
tnpa_embed = get_protein_embedding(tnpa_seq)

traI_clasp = traI_embed[231:271, :]
trwC_clasp = trwC_embed[237:262, :]

print(traI_seq[0:9])
print(trwC_seq[0:9])
print(nes_seq[4:13])
print(tnpa_seq[109:118])

print(len(tnpa_seq))

traI_sDBM = traI_embed[0:9, :]
trwC_sDBM = trwC_embed[0:9, :]
nes_sDBM = nes_embed[4:13, :]
tnpa_sDBM = tnpa_embed[109:118, :]

MMSIAQVRSAGSAGNYYTDKDNYYVLGSMGERWAGRGAEQLGLQGSVDKDVFTRLLEGRLPDGADLSRMQDGSNRHRPGYDLTFSAPKSVSMMAMLGGDKRLIDAHNQAVDFAVRQVEALASTRVMTDGQSETVLTGNLVMALFNHDTSRDQEPQLHTHAVVANVTQHNGEWKTLSSDKVGKTGFIENVYANQIAFGRLYREKLKEQVEALGYETEVVGKHGMWEMPGVPVEAFSGRSQTIREAVGEDASLKSRDVAALDTRKSKQHVDPEIKMAEWMQTLKETGFDIRAYRDAADQRADLRTLTPGPASQDGPDVQQAVTQAIAGLSERKVQFTYTDVLARTVGILPPENGVIERARAGIDEAISREQLIPLDREKGLFTSGIHVLDELSVRALSRDIMKQNRVTVHPEKSVPRTAGYSDAVSVLAQDRPSLAIVSGQGGAAGQRERVAELVMMAREQGREVQIIAADRRSQMNMKQDERLSGELITGRRQLLEGMAFTPGSTVIVDQGEKLSLKETLTLLDGAARHNVQVLITDSGQRTGTGSALMAMKDAGVNTYRWQGGEQRPATIISEPDRNVRYARLAGDFAASVKAGEESVAQVSGVREQAILTQAIRSELKTQGVLGLPEVTMTALSPVWLDSRSRYLRDMYRPGMVMEQWNPETRSHDRYVIDRVTAQSHSLTLRDAQGETQVVRISSLDSSWSLFRPEKMPVADGERLRVTGKIPGLRVSGGDRLQVASVSEDAMTVVVPGRAEPATLPVSDSPFTALKLENGWVETPGHSVSDSATVFASVTQMAMDNATLNGLARSGRDVRLYSSLDETRTAEKLARHPSFTVVSEQIKTRAGETSLETAISHQKSALHTPAQQAIHLALPVVESKKLAFSMVDLLTEAKSFAAEGTGFTELGGEINAQIKRGDLLYVDVAKGYGTGLLVSRASYEAEKSILRHILEGKEAVMPLMERVPGELMEKLTSGQRAATRMILETSDRFTVVQGYAGVGKTT

In [80]:
def query_sDBM(protein_seq, known_sDBM):

    protein_embed = get_protein_embedding(protein_seq)

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    next_start = 0
    next_end = 0 

    for i in range(0, 140):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = i + 1
            next_end = i + 9

    # if ((max_mean - next_max_mean) < 0.01):
    #     print("Unable to suggest sDBM... confidence measure below threshold")
    #     return

  #  diff = ((max_mean - next_max_mean) / max_mean) * 100
    perc = (max_mean - next_max_mean) * 100


    print("sDBM candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"sDBM prediction from {start_seq} to {end_seq}")
    print(f"{next_start} to {next_end}")
    print(protein_seq[start_seq - 1: end_seq])
    print(perc)

In [ ]:
def query_clasp(protein_seq, known_clasp):

    protein_embed = get_protein_embedding(protein_seq)

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    clasp_length, _ = known_clasp.shape

    for i in range(100, 350):
        match_block = protein_embed[i:i+clasp_length, :]

        sim = F.cosine_similarity(match_block, known_clasp, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + clasp_length
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = start_seq
            next_end = end_seq

    if ((max_mean - next_max_mean) < 0.01):
        print("Unable to suggest sDBM... confidence measure below threshold")
        return

  #  diff = ((max_mean - next_max_mean) / max_mean) * 100
    perc = (max_mean - next_max_mean) * 100


    print("Clasp candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"Clasp prediction from {start_seq} to {end_seq}")
    print(f"{next_start} to {next_end}")
    print(protein_seq[start_seq - 1: end_seq])
    print(perc)

In [81]:
query_sDBM(traI_seq, nes_sDBM)

query_sDBM(trwC_seq, nes_sDBM)

query_sDBM(tnpa_seq, nes_sDBM)

query_sDBM(nes_seq, traI_sDBM)

sDBM candidate found!!!
Max Mean Similarity: 0.5529317855834961
Closest Mean Similarity to sDBM Prediction: 0.5051024556159973
sDBM prediction from 1 to 9
3 to 11
MMSIAQVRS
tensor(4.7829, device='cuda:0')
sDBM candidate found!!!
Max Mean Similarity: 0.5743986368179321
Closest Mean Similarity to sDBM Prediction: 0.5094406604766846
sDBM prediction from 1 to 9
6 to 14
MLSHMVLTR
tensor(6.4958, device='cuda:0')
sDBM candidate found!!!
Max Mean Similarity: 0.4507102966308594
Closest Mean Similarity to sDBM Prediction: 0.4496908187866211
sDBM prediction from 50 to 58
0 to 0
ELRVEIIEM
tensor(0.1019, device='cuda:0')
sDBM candidate found!!!
Max Mean Similarity: 0.5865258574485779
Closest Mean Similarity to sDBM Prediction: 0.5615296959877014
sDBM prediction from 10 to 18
7 to 15
FVSKANGQS
tensor(2.4996, device='cuda:0')


In [35]:
query_clasp(traI_seq, trwC_clasp)

Clasp candidate found!!!
Max Mean Similarity: 0.6091692447662354
Closest Mean Similarity to sDBM Prediction: 0.5696449875831604
Clasp prediction from 276 to 300
240 to 264
EWMQTLKETGFDIRAYRDAADQRAD
tensor(3.9524, device='cuda:0')


In [ ]:
query_sDBM(pcv_seq, wdv_sDBM)
query_sDBM(fbnyv_seq, pcv_sDBM)
query_sDBM(fbnyv_seq, wdv_sDBM)